<a href="https://colab.research.google.com/github/rd2080/Pizza-Data-Analysis/blob/main/Data_Science_Assignment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Upload the file directly to Colab
from google.colab import files
uploaded = files.upload()

print("Uploaded files:", list(uploaded.keys()))

Saving Pizza_Sale.xlsx to Pizza_Sale (1).xlsx
Uploaded files: ['Pizza_Sale (1).xlsx']


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import io

# Use the correct uploaded file name
file_name = "Pizza_Sale (1).xlsx"
print(f"Using file: {file_name}")

Using file: Pizza_Sale (1).xlsx


In [ ]:
# Read the uploaded file
xls = pd.ExcelFile(file_name)

# Check available sheet names
print("Available sheets:", xls.sheet_names)

# Read the appropriate sheet
sheet_name = "pizza_sales"  # Adjust if needed based on the output above
df = pd.read_excel(xls, sheet_name=sheet_name)

# Display basic information to confirm the data is loaded correctly
print("\nData loaded. First 5 rows:")
print(df.head())
print("\nData shape:", df.shape)

Available sheets: ['pizza_sales']

Data loaded. First 5 rows:
   pizza_id  order_id  pizza_name_id  quantity           order_date  \
0         1         1     hawaiian_m         1  2015-01-01 00:00:00   
1         2         2  classic_dlx_m         1  2015-01-01 00:00:00   
2         3         2  five_cheese_l         1  2015-01-01 00:00:00   
3         4         2    ital_supr_l         1  2015-01-01 00:00:00   
4         5         2     mexicana_m         1  2015-01-01 00:00:00   

  order_time  unit_price  total_price pizza_size pizza_category  \
0   11:38:36       13.25        13.25          M        Classic   
1   11:57:40       16.00        16.00          M        Classic   
2   11:57:40       18.50        18.50          L         Veggie   
3   11:57:40       20.75        20.75          L        Supreme   
4   11:57:40       16.00        16.00          M         Veggie   

                                   pizza_ingredients  \
0           Sliced Ham, Pineapple, Mozzarella Cheese

In [ ]:
# Create a copy for cleaning
df_clean = df.copy()

# Check missing values
missing_values = df_clean.isnull().sum()
print("\nMissing values by column:")
print(missing_values)

# Data type conversion
df_clean['order_date'] = pd.to_datetime(df_clean['order_date'], errors='coerce')
df_clean['order_time'] = pd.to_datetime(df_clean['order_time'], format='%H:%M:%S', errors='coerce')

# Data validation
df_clean['calculated_total'] = df_clean['unit_price'] * df_clean['quantity']
price_diff = (df_clean['calculated_total'] - df_clean['total_price']).abs()
inconsistent_prices = df_clean[price_diff > 0.01]
print(f"\nNumber of inconsistent prices: {len(inconsistent_prices)}")

# Feature engineering
df_clean['day_of_week'] = df_clean['order_date'].dt.day_name()
df_clean['month'] = df_clean['order_date'].dt.month
df_clean['hour'] = df_clean['order_time'].dt.hour
df_clean['time_of_day'] = pd.cut(
    df_clean['hour'],
    bins=[0, 11, 15, 19, 24],
    labels=['Morning', 'Afternoon', 'Evening', 'Night']
)

# Handle missing values
# Instead of dropping, fill with appropriate values where possible
df_clean['pizza_category'] = df_clean['pizza_category'].fillna('Unknown')
df_clean = df_clean.dropna(subset=['total_price', 'pizza_name'])  # Only drop where essential

# Standardize categorical values - fix the str methods
df_clean['pizza_category'] = df_clean['pizza_category'].astype(str).str.strip().str.lower()
df_clean['pizza_size'] = df_clean['pizza_size'].astype(str).str.strip().str.upper()

# Outlier handling (more nuanced)
for col in ['quantity', 'unit_price', 'total_price']:
    upper_limit = df_clean[col].quantile(0.995)  # Less strict than z-score
    print(f"Upper limit for {col}: {upper_limit}")
    df_clean = df_clean[df_clean[col] <= upper_limit]


Missing values by column:
pizza_id              0
order_id              0
pizza_name_id        16
quantity              0
order_date            0
order_time            0
unit_price            0
total_price           7
pizza_size            0
pizza_category       23
pizza_ingredients    13
pizza_name            7
dtype: int64

Number of inconsistent prices: 0
Upper limit for quantity: 2.0
Upper limit for unit_price: 25.5
Upper limit for total_price: 40.5


<ipython-input-12-23fadc20823f>:35: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_clean['pizza_category'] = df_clean['pizza_category'].astype(str).str.strip().str.lower()
<ipython-input-12-23fadc20823f>:36: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_clean['pizza_size'] = df_clean['pizza_size'].astype(str).str.strip().str.upper()


In [ ]:
# Create analysis-ready datasets
# 1. Time-based analysis
time_sales = df_clean.groupby([df_clean['order_date'].dt.date, 'time_of_day'])['total_price'].sum().reset_index()

# 2. Pizza popularity
pizza_popularity = df_clean.groupby(['pizza_name', 'pizza_size'])['quantity'].sum().reset_index()
pizza_popularity = pizza_popularity.sort_values('quantity', ascending=False)
print("\nTop 5 most popular pizzas:")
print(pizza_popularity.head())

# 3. Category analysis
category_analysis = df_clean.groupby(['pizza_category', 'pizza_size'])[['quantity', 'total_price']].sum().reset_index()
print("\nSales by category and size:")
print(category_analysis)

# 4. Ingredient analysis
def extract_ingredients(ingredients_str):
    if pd.isna(ingredients_str):
        return []
    return [ing.strip() for ing in ingredients_str.split(',')]

all_ingredients = []
for ing_list in df_clean['pizza_ingredients'].apply(extract_ingredients):
    all_ingredients.extend(ing_list)

ingredient_counts = pd.Series(all_ingredients).value_counts().reset_index()
ingredient_counts.columns = ['ingredient', 'count']
print("\nTop 10 most common ingredients:")
print(ingredient_counts.head(10))

# 5. Daily and hourly order patterns
daily_orders = df_clean.groupby([df_clean['order_date'].dt.date])['order_id'].nunique().reset_index()
daily_orders.columns = ['date', 'num_orders']

hourly_orders = df_clean.groupby(['hour'])['order_id'].nunique().reset_index()
hourly_orders.columns = ['hour', 'num_orders']
print("\nOrder patterns by hour (top 5 busiest hours):")
print(hourly_orders.sort_values('num_orders', ascending=False).head())

# 6. Average items per order
orders_items = df_clean.groupby('order_id')['quantity'].sum().reset_index()
avg_items_per_order = orders_items['quantity'].mean()
print(f"\nAverage items per order: {avg_items_per_order}")

# 7. Seasonal analysis
df_clean['month_name'] = df_clean['order_date'].dt.month_name()
monthly_sales = df_clean.groupby('month_name')['total_price'].sum().reset_index()
print("\nMonthly sales:")
print(monthly_sales)

# 8. Price point analysis
df_clean['price_category'] = pd.cut(
    df_clean['unit_price'],
    bins=[0, 12, 18, 25, 100],
    labels=['Budget', 'Standard', 'Premium', 'Luxury']
)
price_category_sales = df_clean.groupby('price_category')[['quantity', 'total_price']].sum().reset_index()
print("\nSales by price category:")
print(price_category_sales)

<ipython-input-13-e1cd8e216d66>:3: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  time_sales = df_clean.groupby([df_clean['order_date'].dt.date, 'time_of_day'])['total_price'].sum().reset_index()



Top 5 most popular pizzas:
                  pizza_name pizza_size  quantity
3         The Big Meat Pizza          S      1893
20     The Five Cheese Pizza          L      1405
84    The Thai Chicken Pizza          L      1321
21     The Four Cheese Pizza          L      1316
18  The Classic Deluxe Pizza          M      1175

Sales by category and size:
   pizza_category pizza_size  quantity  total_price
0         chicken          L      4668     96861.00
1         chicken          M      3883     65040.25
2         chicken          S      2223     28343.25
3         classic          L      3996     73289.25
4         classic          M      4099     60387.75
5         classic          S      6110     69529.75
6         classic         XL       536     13668.00
7         supreme          L      4426     91395.50
8         supreme          M      4037     66328.25
9         supreme          S      3352     46917.10
10        unknown          L        10       199.75
11        unknown  

<ipython-input-13-e1cd8e216d66>:57: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  price_category_sales = df_clean.groupby('price_category')[['quantity', 'total_price']].sum().reset_index()


In [ ]:
# Create a summary stats file
summary = {
    'total_orders': df_clean['order_id'].nunique(),
    'total_pizzas_sold': df_clean['quantity'].sum(),
    'total_revenue': df_clean['total_price'].sum(),
    'avg_order_value': df_clean.groupby('order_id')['total_price'].sum().mean(),
    'avg_items_per_order': avg_items_per_order,
    'most_popular_pizza': pizza_popularity.iloc[0]['pizza_name'] + ' (' + pizza_popularity.iloc[0]['pizza_size'] + ')',
    'most_popular_category': category_analysis.groupby('pizza_category')['quantity'].sum().idxmax(),
    'busiest_day_of_week': df_clean.groupby('day_of_week')['order_id'].nunique().idxmax(),
    'busiest_hour': hourly_orders.sort_values('num_orders', ascending=False).iloc[0]['hour'],
    'busiest_month': monthly_sales.sort_values('total_price', ascending=False).iloc[0]['month_name'],
    'most_common_ingredient': ingredient_counts.iloc[0]['ingredient']
}

# Convert the summary to a DataFrame for better display
summary_df = pd.DataFrame({'Metric': list(summary.keys()), 'Value': list(summary.values())})
print("\nSummary Statistics:")
print(summary_df)

# Save files for download
df_clean.to_excel("Cleaned_Pizza_Sales_Full.xlsx", index=False)
time_sales.to_excel("time_sales_analysis.xlsx", index=False)
pizza_popularity.to_excel("pizza_popularity.xlsx", index=False)
category_analysis.to_excel("category_analysis.xlsx", index=False)
ingredient_counts.to_excel("ingredient_popularity.xlsx", index=False)
daily_orders.to_excel("daily_orders.xlsx", index=False)
hourly_orders.to_excel("hourly_orders.xlsx", index=False)
monthly_sales.to_excel("monthly_sales.xlsx", index=False)
price_category_sales.to_excel("price_category_sales.xlsx", index=False)
summary_df.to_excel("sales_summary.xlsx", index=False)

print("\nData cleaning and preparation complete. All analysis files created.")


Summary Statistics:
                    Metric                   Value
0             total_orders                   21323
1        total_pizzas_sold                   48979
2            total_revenue               805406.95
3          avg_order_value               37.771746
4      avg_items_per_order                2.297003
5       most_popular_pizza  The Big Meat Pizza (S)
6    most_popular_category                 classic
7      busiest_day_of_week                  Friday
8             busiest_hour                      12
9            busiest_month                 January
10  most_common_ingredient                  Garlic

Data cleaning and preparation complete. All analysis files created.


In [ ]:
# --- DATA WRANGLING SECTION ---
# This section covers additional data preparation techniques beyond basic cleaning

# 1. Feature Creation: Calculate time-since-open and derive additional metrics
# Let's assume the store opens at 10 AM
opening_hour = 10
df_clean['minutes_since_opening'] = (df_clean['hour'] - opening_hour) * 60 + df_clean['order_time'].dt.minute
df_clean['is_weekend'] = df_clean['order_date'].dt.dayofweek >= 5  # 5,6 are weekend days (Sat, Sun)

# 2. Pizza Name Parsing: Extract pizza type from name for finer categorization
df_clean['pizza_type'] = df_clean['pizza_name'].str.replace('The ', '').str.split(' Pizza').str[0]

# 3. Order Sequence Analysis: Add sequence number of orders within day
df_clean['order_date_only'] = df_clean['order_date'].dt.date
df_clean['order_sequence'] = df_clean.groupby('order_date_only')['order_time'].rank(method='dense')

# 4. Calculating pizza complexity (number of ingredients)
df_clean['ingredient_count'] = df_clean['pizza_ingredients'].apply(
    lambda x: len(str(x).split(',')) if pd.notna(x) else 0
)

# 5. Create size-based metrics
size_multiplier = {'S': 1, 'M': 1.5, 'L': 2, 'XL': 3}
df_clean['size_factor'] = df_clean['pizza_size'].map(size_multiplier)
df_clean['adjusted_quantity'] = df_clean['quantity'] * df_clean['size_factor']

# 6. Revenue per ingredient
df_clean['revenue_per_ingredient'] = df_clean.apply(
    lambda row: row['total_price'] / row['ingredient_count'] if row['ingredient_count'] > 0 else 0,
    axis=1
)

# 7. Create customer value metrics (based on order_id as proxy for customer)
order_total = df_clean.groupby('order_id')['total_price'].sum().reset_index()
order_total.columns = ['order_id', 'order_total']
df_clean = pd.merge(df_clean, order_total, on='order_id')

# 8. Calculate percentage of order (how much of the total order does each pizza represent)
df_clean['pct_of_order'] = df_clean['total_price'] / df_clean['order_total'] * 100

# 9. Time-based categorization
df_clean['part_of_day'] = pd.cut(
    df_clean['hour'],
    bins=[0, 6, 11, 14, 17, 21, 24],
    labels=['Night', 'Early Morning', 'Lunch', 'Afternoon', 'Dinner', 'Late Night']
)

# 10. Calculate sales velocity (items per hour)
hourly_sales = df_clean.groupby(['order_date_only', 'hour'])['quantity'].sum().reset_index()
hourly_sales.columns = ['date', 'hour', 'items_sold']
hourly_sales['sales_velocity'] = hourly_sales['items_sold'] / 1  # items per hour

# 11. Time series features
df_clean['day_of_month'] = df_clean['order_date'].dt.day
df_clean['week_of_year'] = df_clean['order_date'].dt.isocalendar().week

# 12. Advanced analysis: Create basket affinity data
# Group pizzas by order to see which pizzas are commonly ordered together
order_baskets = df_clean.groupby('order_id')['pizza_name'].apply(list).reset_index()
order_baskets.columns = ['order_id', 'basket']

# 13. Price tier relative to category
category_avg_price = df_clean.groupby('pizza_category')['unit_price'].mean().reset_index()
category_avg_price.columns = ['pizza_category', 'category_avg_price']
df_clean = pd.merge(df_clean, category_avg_price, on='pizza_category')
df_clean['price_relative_to_category'] = df_clean['unit_price'] / df_clean['category_avg_price']

# 14. Create a normalized total price (per ingredient)
df_clean['price_per_ingredient'] = df_clean['unit_price'] / df_clean['ingredient_count']

# 15. Additional cross-categorical analysis
category_size_metrics = df_clean.groupby(['pizza_category', 'pizza_size']).agg({
    'total_price': ['sum', 'mean'],
    'quantity': 'sum',
    'ingredient_count': 'mean',
    'revenue_per_ingredient': 'mean'
}).reset_index()

# Flatten the MultiIndex in the columns
category_size_metrics.columns = ['_'.join(col).strip('_') for col in category_size_metrics.columns.values]

# 16. Save the enhanced dataset
df_enhanced = df_clean.copy()
df_enhanced.to_excel("Enhanced_Pizza_Sales.xlsx", index=False)

# 17. Create basket analysis dataset
basket_analysis = order_baskets.copy()
basket_analysis.to_excel("Pizza_Basket_Analysis.xlsx", index=False)

# 18. Create hourly performance dataset
hourly_performance = hourly_sales.copy()
hourly_performance.to_excel("Hourly_Sales_Velocity.xlsx", index=False)

# 19. Create category performance comparison
category_performance = category_size_metrics.copy()
category_performance.to_excel("Category_Size_Performance.xlsx", index=False)

print("\nAdvanced data wrangling complete. Enhanced datasets created.")
print("Additional analysis files saved:")
print("- Enhanced_Pizza_Sales.xlsx: Contains all derived features")
print("- Pizza_Basket_Analysis.xlsx: For analyzing commonly ordered combinations")
print("- Hourly_Sales_Velocity.xlsx: Time-based sales rates")
print("- Category_Size_Performance.xlsx: Detailed category performance metrics")


Advanced data wrangling complete. Enhanced datasets created.
Additional analysis files saved:
- Enhanced_Pizza_Sales.xlsx: Contains all derived features
- Pizza_Basket_Analysis.xlsx: For analyzing commonly ordered combinations
- Hourly_Sales_Velocity.xlsx: Time-based sales rates
- Category_Size_Performance.xlsx: Detailed category performance metrics


In [ ]:
from google.colab import files

# Download the main cleaned and enhanced datasets
files.download("Cleaned_Pizza_Sales_Full.xlsx")
files.download("Enhanced_Pizza_Sales.xlsx")

# Download the analysis files
files.download("pizza_popularity.xlsx")
files.download("category_analysis.xlsx")
files.download("ingredient_popularity.xlsx")
files.download("hourly_orders.xlsx")
files.download("monthly_sales.xlsx")
files.download("price_category_sales.xlsx")
files.download("sales_summary.xlsx")

# Download the additional advanced analysis files
files.download("Pizza_Basket_Analysis.xlsx")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>